# 03_fredf_baseline

Прототип реализации FreDF и сравнение с базовыми моделями.

# Применение моделей на исходных данных

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
plt.rcParams['figure.dpi'] = 300

from ts_toolkit.io import clean_timeseries
from ts_toolkit.calendar import add_hour_sin_cos
from ts_toolkit.viz import plot_history_forecast
from ts_toolkit.split import three_way_split
from src.models.clown_leakage_model import DelayForecastModel
from ts_toolkit.metrics import daily_mae
from ts_toolkit.metrics import global_metrics

In [ ]:
from src.data_loader import fetch_frame
# df = fetch_frame()
df = fetch_frame(
    start_date="2024-11-25 18:00:00",
    end_date  ="2024-12-11 12:10:00",
    use_cache=True,             # всегда тянуть свежие данные
    cache_filename="common_cad_avg1h_20241125_20241211.parquet",
    # verbose=True                 # чтобы видеть прогресс
)
df.rename(columns={'common_cad_avg1h_instance_consumer_from_ws_with_metrics:8000_job_consumer_from_ws_with_metrics_service_castle': 'common_cad_avg1h'}, inplace=True)

print(df.head())

print(df.info())

print(df.describe())

print(df.isnull().sum())


In [ ]:
df.info()

df.describe()

df.isnull().sum()

df.head()

df.tail()

In [4]:
df = clean_timeseries(df, 'common_cad_avg1h')   # вместо блока 0‑3
df = add_hour_sin_cos(df)
feature_cols = ['hour_sin', 'hour_cos']

In [ ]:
feature_cols

In [ ]:
# инициализируем — можно менять lags/roll_windows/test_size
model = DelayForecastModel(
    horizon       = 900,   # сутки = 24*60*60 / 15 = 5760 точек
    test_size     = 0.01,    # 80 % train, 20 % hold-out
    lags          = [96, 192, 5760],
    roll_windows  = [4,96,192]
    # lags          = [1, 2, 4, 96, 192, 5760],
    # roll_windows  = [4,96,192,1920,2880,4320,5760,8640]
)


# fit + автоматически нарисует график «train / test-true / test-pred»
train_df, test_df = model.fit(
    df,
    target_col   = 'common_cad_avg1h',
    feature_cols = feature_cols,      # если есть другие метрики — впишите их названия здесь
    plot         = True
)

In [ ]:
print(len(train_df), len(test_df))

In [ ]:
# берём последние (max(lags, roll_windows) + horizon) точек
history_needed = max(model.lags + model.roll_windows) + model.horizon
df_last = df.tail(history_needed)

# готовим фичи и предсказываем
df_future = model.prepare_future(df_last, 'common_cad_avg1h')

y_hat_next_day = model.predict(df_future)[-model.horizon:]
future_index   = pd.date_range(
    start=df.index[-1] + pd.Timedelta(seconds=15),
    periods=model.horizon,
    freq='15S'
)

from ts_toolkit.viz import plot_history_forecast

# ----- визуализация прогноза -----
history_series = df_last['common_cad_avg1h'].iloc[-3*5760:]
forecast_series = pd.Series(y_hat_next_day, index=future_index)

plot_history_forecast(
    history=history_series,
    forecast=forecast_series,
    title='p90 latency — history vs 24 h forecast'
)


## 2. Код-шаблоны для подробного анализа


In [ ]:
# ---------------------------------------------------
# 2.1  получить true & pred на тесте
# ---------------------------------------------------
#  полный список признаков, который модель видела
feat_cols = model.model.feature_names_          # ровно в том порядке!
test_feat  = test_df[feat_cols]                 # никаких пропусков
assert list(test_feat.columns) == list(model.model.feature_names_)

# 1. истинные значения и прогноз
test_true  = test_df['common_cad_avg1h']
test_pred  = pd.Series(
    model.model.predict(test_feat),
    index=test_true.index,
    name='pred'
)

# 2. остатки
resid = test_true - test_pred


# ---------------------------------------------------
# 2.2  сводные метрики
# ---------------------------------------------------
metrics = global_metrics(test_true, test_pred)
metrics_df = pd.DataFrame([metrics]).T.rename(columns={0: "value"})
print("\n*** Hold-out metrics ***")
print(metrics_df)

# ---------------------------------------------------
# 2.3  метрики по суткам
# ---------------------------------------------------
daily_mae_result = daily_mae(test_true, test_pred)
print("\nMAE by day:")
print(daily_mae_result.tail())

# ---------------------------------------------------
# 2.4  распределение ошибок
# ---------------------------------------------------
plt.figure(figsize=(12,4))
plt.hist(resid, bins=100, alpha=.7, edgecolor='black')
plt.axvline(resid.mean(), color='r', linestyle='--', label=f"mean={resid.mean():.1f}")
plt.title("Residual distribution on test")
plt.xlabel("error (true − pred)")
plt.legend(); plt.tight_layout(); plt.show()

# ---------------------------------------------------
# 2.5  true vs pred scatter
# ---------------------------------------------------
plt.figure(figsize=(6,6))
plt.scatter(test_true, test_pred, s=3, alpha=0.5)
lim = [0, max(test_true.max(), test_pred.max())*1.05]
plt.plot(lim, lim, 'k--')
plt.xlabel("true"); plt.ylabel("pred")
plt.title("True vs predicted, test split")
plt.tight_layout(); plt.show()


## 3. Feature Importance (какие лаги реально работают)

In [ ]:
feat_names = model.model.feature_names_        # ← то, что CatBoost запомнил
importances = model.model.get_feature_importance(type='FeatureImportance')

# в один датафрейм
imp_df = (pd.DataFrame({"feature": feat_names,
                        "importance": importances})
            .sort_values(by="importance", ascending=False)
            .reset_index(drop=True))

# ---------------------------------------------------
#  топ-20 на графике
# ---------------------------------------------------
plt.figure(figsize=(10,6))
plt.barh(imp_df.feature.head(20)[::-1],
         imp_df.importance.head(20)[::-1])
plt.title("Top-20 feature importances (CatBoost)")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(14,4))
plt.plot(resid.index, resid, alpha=0.7)
plt.title("Residuals over time (test split)")
plt.axhline(0, color='k', lw=1)
plt.tight_layout(); plt.show()


# Проверка на реальных данных

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.  выбираем точку отсечки (24 h до конца ряда)
# ─────────────────────────────────────────────────────────────
step       = pd.Timedelta(seconds=15)
H          = model.horizon                   # 5760
cut_time   = df.index[-H-1]                  # last seen by model

hist_need  = max(model.lags + model.roll_windows) + 10
df_hist    = df.loc[:cut_time].tail(hist_need)

# ─────────────────────────────────────────────────────────────
# 2.  генерируем признаки и прогноз
# ─────────────────────────────────────────────────────────────
df_future  = model.prepare_future(df_hist, 'common_cad_avg1h')
y_pred     = model.model.predict(df_future)

pred_idx   = df_future.index            # ровно столько, сколько точек в y_pred
pred_ser   = pd.Series(y_pred, index=pred_idx, name='pred')

# ─────────────────────────────────────────────────────────────
# 3.  реальные значения (align по pred_idx)
# ─────────────────────────────────────────────────────────────
y_true = df['common_cad_avg1h'].loc[pred_idx]   # факты под теми же метками

# ─────────────────────────────────────────────────────────────
# 4.  метрики
# ─────────────────────────────────────────────────────────────
metrics = global_metrics(y_true, pred_ser)
print(f"Blind 24-h test • MAE={metrics['MAE']:.1f}  RMSE={metrics['RMSE']:.1f}  MAPE={metrics['MAPE']:.2f}%  "
      f"on {len(y_true)} valid points")

# ─────────────────────────────────────────────────────────────
# 5.  график: 3 сут истории + пред / факт
# ─────────────────────────────────────────────────────────────

hist_start = cut_time - pd.Timedelta(hours=72)
plot_history_forecast(
    history  = df.loc[hist_start:cut_time, 'common_cad_avg1h'],
    forecast = pred_ser,
    actual   = y_true,
    title    = 'Blind forecast vs actual — last 24 h'
)


In [ ]:
print("train_END :", train_df.index[-1])
print("cut_time  :", cut_time)
print("cut_time > train_END ?", cut_time > train_df.index[-1])


# Максимально честный тест

In [15]:
df_train, df_val, df_hold = three_way_split(df, train_ratio=0.7, val_ratio=0.2)

# ── fit на train, early-stop на val ───────────────────────────
model = DelayForecastModel(
    horizon       = 5760,   # сутки = 24*60*60 / 15 = 5760 точек
    test_size     = 0.2,    # 80 % train, 20 % hold-out
    lags          = [1, 2, 4, 96, 192, 5760],
    # roll_windows  = [96, 384, 5760]          # 24 мин, 1 ч, 1 сут
    roll_windows  = [4,96,192,1920,2880,4320,5760,8640]
)
train_df, _ = model.fit(
    df_train, 'common_cad_avg1h',
    feature_cols=['hour_sin','hour_cos'],
    plot=False
)

# ── прогноз на hold-out (слепой) ──────────────────────────────
history_need = max(model.lags + model.roll_windows) + 10
df_hist = pd.concat([df_val, df_hold]).tail(history_need)   # контекст

df_future = model.prepare_future(df_hist, 'common_cad_avg1h')
y_pred    = model.model.predict(df_future)
pred_idx  = df_future.index

y_true    = df_hold['common_cad_avg1h'].reindex(pred_idx).dropna()
y_pred    = pd.Series(y_pred, index=pred_idx).loc[y_true.index]

# print("MAE =", mean_absolute_error(y_true, y_pred))


In [ ]:
# pred_idx и y_pred уже получены в предыдущем шаге
y_true = df['common_cad_avg1h'].loc[pred_idx]

metrics = global_metrics(y_true, y_pred)
print(f"Blind 24-h test • MAE={metrics['MAE']:.1f}  RMSE={metrics['RMSE']:.1f}  MAPE={metrics['MAPE']:.2f}%  "
      f"on {len(y_true)} valid points")

# ── график: история 3 сут + прогноз vs факт ──
hist_start = pred_idx[0] - pd.Timedelta(hours=24)
plot_history_forecast(
    history  = df.loc[hist_start:pred_idx[0], 'common_cad_avg1h'],
    forecast = pd.Series(y_pred, index=pred_idx),
    actual   = y_true,
    title    = 'Blind forecast vs actual — hold-out 24 h'
)


In [ ]:
from ts_toolkit.metrics import daily_mae

daily_mae_df = daily_mae(y_true, y_pred)
print(daily_mae_df)